# 05 Modeling with Safe MLflow Logging

**Project:** AI-Powered Customer Retention Decision Support System for E-Commerce  
**Input table:** `customer_modeling_90d.csv`  
**Target:** `inactive_90d`  
**Goal:** Compare classification models, tune a decision threshold, and optionally log selected results to MLflow.

This notebook is designed to be safe to rerun. Existing MLflow runs and saved results are preserved unless
logging or saving is explicitly enabled.

## Safety Controls

The notebook has three separate side-effect controls:

- `SAVE_OUTPUTS = False`: do not overwrite CSV outputs such as `model_results_90d.csv`,
  `customer_risk_scores_test.csv`, or `threshold_tuning_results_90d.csv`.
- `LOG_MODEL_COMPARISON_TO_MLFLOW = False`: do not create MLflow runs while comparing candidate models.
- `LOG_FINAL_MODEL_TO_MLFLOW = False`: do not log the final tuned model to MLflow.

Keep these as `False` for normal reruns. Change them only when you intentionally want to regenerate saved
files or create new MLflow runs.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    HAS_PLOTS = True
    sns.set_theme(
        style="whitegrid",
        context="notebook",
        palette="Set2",
        rc={
            "figure.figsize": (11, 5),
            "axes.titleweight": "bold",
            "axes.titlesize": 14,
            "axes.labelsize": 11,
            "legend.frameon": False,
        },
    )
except ImportError:
    HAS_PLOTS = False
    print("Plotting libraries are not available in this environment; chart cells will be skipped.")

try:
    import mlflow
    import mlflow.sklearn
    from mlflow.models.signature import infer_signature

    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False
    mlflow = None
    infer_signature = None
    print("MLflow is not installed in this environment. Modeling can still run, but logging is unavailable.")

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    XGBClassifier = None
    print("XGBoost is not installed. The XGBoost candidate will be skipped.")

## 2. Paths and Rerun Controls

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "notebooks"
    else Path(r"C:/Learning/BANA8083/AI-retention-decision-support")
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MLFLOW_TRACKING_DIR = MODELS_DIR / "mlruns"

MODELING_PATH = PROCESSED_DIR / "customer_modeling_90d.csv"
MODEL_RESULTS_PATH = PROCESSED_DIR / "model_results_90d.csv"
RISK_SCORES_PATH = PROCESSED_DIR / "customer_risk_scores_test.csv"
THRESHOLD_RESULTS_PATH = PROCESSED_DIR / "threshold_tuning_results_90d.csv"

SAVE_OUTPUTS = False
LOG_MODEL_COMPARISON_TO_MLFLOW = False
LOG_FINAL_MODEL_TO_MLFLOW = False

RANDOM_STATE = 42
TEST_SIZE = 0.25
SELECTED_THRESHOLD = 0.40
EXPERIMENT_NAME = "customer_inactivity_prediction_90d"

paths = pd.DataFrame(
    {
        "asset": [
            "modeling dataset",
            "saved model results",
            "saved risk scores",
            "saved threshold results",
            "MLflow tracking directory",
        ],
        "path": [
            MODELING_PATH,
            MODEL_RESULTS_PATH,
            RISK_SCORES_PATH,
            THRESHOLD_RESULTS_PATH,
            MLFLOW_TRACKING_DIR,
        ],
        "exists": [
            MODELING_PATH.exists(),
            MODEL_RESULTS_PATH.exists(),
            RISK_SCORES_PATH.exists(),
            THRESHOLD_RESULTS_PATH.exists(),
            MLFLOW_TRACKING_DIR.exists(),
        ],
    }
)
paths

In [ ]:
if (LOG_MODEL_COMPARISON_TO_MLFLOW or LOG_FINAL_MODEL_TO_MLFLOW) and not HAS_MLFLOW:
    raise ImportError("MLflow logging was enabled, but mlflow is not installed in this environment.")

if HAS_MLFLOW and (LOG_MODEL_COMPARISON_TO_MLFLOW or LOG_FINAL_MODEL_TO_MLFLOW):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(f"file:{MLFLOW_TRACKING_DIR}")
    mlflow.set_experiment(EXPERIMENT_NAME)
    print("MLflow logging is enabled.")
    print("MLflow tracking URI:", mlflow.get_tracking_uri())
else:
    print("MLflow logging is disabled. Existing MLflow runs will not be changed.")

## 3. Load Modeling Data

In [ ]:
customer_model = pd.read_csv(
    MODELING_PATH,
    parse_dates=["first_purchase_date", "last_purchase_date"],
)

print(f"Rows: {len(customer_model):,}")
print(f"Columns: {customer_model.shape[1]:,}")
print(f"Inactive customers: {customer_model['inactive_90d'].sum():,}")
print(f"Active customers: {customer_model['active_90d'].sum():,}")
customer_model.head()

In [ ]:
data_quality = pd.DataFrame(
    {
        "check": [
            "rows",
            "unique_customers",
            "duplicate_customer_rows",
            "missing_target",
            "inactive_rate",
        ],
        "value": [
            len(customer_model),
            customer_model["CustomerID"].nunique(),
            customer_model["CustomerID"].duplicated().sum(),
            customer_model["inactive_90d"].isna().sum(),
            customer_model["inactive_90d"].mean(),
        ],
    }
)
data_quality

## 4. Feature Matrix and Train/Test Split

In [ ]:
TARGET = "inactive_90d"

drop_cols = [
    "CustomerID",
    "first_purchase_date",
    "last_purchase_date",
    "active_90d",
    "inactive_90d",
]

X = customer_model.drop(columns=drop_cols)
y = customer_model[TARGET]

categorical_cols = ["primary_country"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

print("X shape:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True).rename("share"))
print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(X_train), len(X_test)],
        "inactive_rate": [y_train.mean(), y_test.mean()],
    }
)
split_summary

## 5. Preprocessing Pipeline

In [ ]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)
preprocessor

## 6. Model Candidates

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=RANDOM_STATE,
    ),
}

if XGBOOST_AVAILABLE:
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
    )

pd.DataFrame({"model_name": list(models.keys())})

## 7. Train, Evaluate, and Optionally Log Candidate Models

In [ ]:
def build_pipeline(classifier):
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )


def metric_dict(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan,
    }


def evaluate_model(model_name, classifier):
    model = build_pipeline(clone(classifier))
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    metrics = metric_dict(y_test, y_pred, y_proba)

    if LOG_MODEL_COMPARISON_TO_MLFLOW:
        with mlflow.start_run(run_name=model_name):
            mlflow.log_param("model_name", model_name)
            mlflow.log_param("target", TARGET)
            mlflow.log_param("test_size", TEST_SIZE)
            mlflow.log_param("random_state", RANDOM_STATE)

            for param_name, param_value in classifier.get_params().items():
                mlflow.log_param(param_name, param_value)

            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(metric_name, metric_value)

            mlflow.sklearn.log_model(model, artifact_path="model")

    print("=" * 60)
    print(f"Model: {model_name}")
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name}: {metric_value:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return {
        "model_name": model_name,
        **metrics,
    }

In [ ]:
all_results = []

for model_name, classifier in models.items():
    result = evaluate_model(model_name, classifier)
    all_results.append(result)

model_results = (
    pd.DataFrame(all_results)
    .sort_values(by=["roc_auc", "f1_score", "recall"], ascending=False)
    .reset_index(drop=True)
)

model_results

## 8. Compare Against Saved Results

In [ ]:
if MODEL_RESULTS_PATH.exists():
    saved_model_results = pd.read_csv(MODEL_RESULTS_PATH)
    saved_model_results
else:
    saved_model_results = pd.DataFrame()
    print("No saved model_results_90d.csv found.")

In [ ]:
if not saved_model_results.empty:
    result_check = saved_model_results.merge(
        model_results,
        on="model_name",
        how="outer",
        suffixes=("_saved", "_rerun"),
    )
    result_check["roc_auc_delta"] = result_check["roc_auc_rerun"] - result_check["roc_auc_saved"]
    result_check["f1_delta"] = result_check["f1_score_rerun"] - result_check["f1_score_saved"]
    result_check[
        [
            "model_name",
            "roc_auc_saved",
            "roc_auc_rerun",
            "roc_auc_delta",
            "f1_score_saved",
            "f1_score_rerun",
            "f1_delta",
        ]
    ]
else:
    model_results

In [ ]:
if HAS_PLOTS:
    plot_results = model_results.sort_values("roc_auc", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=plot_results, x="roc_auc", y="model_name", color="#4C78A8", ax=ax)
    ax.set_title("Model Comparison by ROC-AUC")
    ax.set_xlabel("ROC-AUC")
    ax.set_ylabel("")
    ax.set_xlim(max(0.0, plot_results["roc_auc"].min() - 0.03), 1.0)
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart because plotting libraries are not available.")

## 9. Optional Save: Model Comparison

In [ ]:
if SAVE_OUTPUTS:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    model_results.to_csv(MODEL_RESULTS_PATH, index=False)
    print(f"Saved model results because SAVE_OUTPUTS=True: {MODEL_RESULTS_PATH}")
else:
    print("SAVE_OUTPUTS=False, so model_results_90d.csv was not written or overwritten.")

## 10. Refit Best Model In Memory

In [ ]:
best_model_name = model_results.iloc[0]["model_name"]
best_classifier = clone(models[best_model_name])

best_model = build_pipeline(best_classifier)
best_model.fit(X_train, y_train)

y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred_default = best_model.predict(X_test)

default_metrics = metric_dict(y_test, y_pred_default, y_proba)

print("Best model from this run:", best_model_name)
print("Default threshold metrics:")
for metric_name, metric_value in default_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

## 11. Threshold Tuning

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
threshold_rows = []

for threshold in thresholds:
    y_pred_threshold = (y_proba >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": threshold,
            "accuracy": accuracy_score(y_test, y_pred_threshold),
            "precision": precision_score(y_test, y_pred_threshold, zero_division=0),
            "recall": recall_score(y_test, y_pred_threshold, zero_division=0),
            "f1_score": f1_score(y_test, y_pred_threshold, zero_division=0),
            "predicted_at_risk_customers": y_pred_threshold.sum(),
            "predicted_at_risk_rate": y_pred_threshold.mean(),
        }
    )

threshold_results = pd.DataFrame(threshold_rows)
threshold_results.sort_values("f1_score", ascending=False).head(10)

In [ ]:
if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.lineplot(data=threshold_results, x="threshold", y="precision", marker="o", label="Precision", ax=ax)
    sns.lineplot(data=threshold_results, x="threshold", y="recall", marker="o", label="Recall", ax=ax)
    sns.lineplot(data=threshold_results, x="threshold", y="f1_score", marker="o", label="F1-score", ax=ax)
    ax.axvline(SELECTED_THRESHOLD, color="#555555", linestyle="--", linewidth=1)
    ax.set_title("Threshold Tradeoff for At-Risk Customer Prediction")
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Metric value")
    ax.set_ylim(0, 1)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart because plotting libraries are not available.")

## 12. Final Tuned Threshold Evaluation

In [ ]:
y_pred_tuned = (y_proba >= SELECTED_THRESHOLD).astype(int)

final_metrics = metric_dict(y_test, y_pred_tuned, y_proba)

print("Selected threshold:", SELECTED_THRESHOLD)
for metric_name, metric_value in final_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned, zero_division=0))

**Interpretation:** The selected threshold should reflect the retention use case. When the business goal is to
find at-risk customers for outreach, recall is usually important because missing likely inactive customers can
mean missed retention opportunities. Precision still matters because campaigns have cost.

## 13. Risk Scores and Optional Output Files

In [ ]:
test_predictions = customer_model.loc[X_test.index, ["CustomerID"]].copy()
test_predictions["actual_inactive_90d"] = y_test.values
test_predictions["predicted_inactive_probability"] = y_proba
test_predictions["predicted_inactive_90d"] = y_pred_tuned
test_predictions["selected_threshold"] = SELECTED_THRESHOLD

test_predictions = test_predictions.sort_values(
    "predicted_inactive_probability",
    ascending=False,
).reset_index(drop=True)

test_predictions.head(10)

In [ ]:
if SAVE_OUTPUTS:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    test_predictions.to_csv(RISK_SCORES_PATH, index=False)
    threshold_results.to_csv(THRESHOLD_RESULTS_PATH, index=False)
    print(f"Saved risk scores because SAVE_OUTPUTS=True: {RISK_SCORES_PATH}")
    print(f"Saved threshold results because SAVE_OUTPUTS=True: {THRESHOLD_RESULTS_PATH}")
else:
    print("SAVE_OUTPUTS=False, so risk scores and threshold results were not written or overwritten.")

## 14. Optional Final MLflow Logging

In [ ]:
threshold_info = {
    "selected_threshold": float(SELECTED_THRESHOLD),
    "threshold_reason": "Selected to support recall-oriented customer retention outreach",
    "business_goal": "Identify customers likely to become inactive in the next 90 days",
}

if LOG_FINAL_MODEL_TO_MLFLOW:
    if mlflow.active_run() is not None:
        mlflow.end_run()

    signature = infer_signature(X_test, best_model.predict(X_test))
    input_example = X_test.head(5)

    with mlflow.start_run(run_name=f"Final {best_model_name} Tuned Threshold"):
        mlflow.log_param("model_name", best_model_name)
        mlflow.log_param("target", TARGET)
        mlflow.log_param("prediction_window_days", 90)
        mlflow.log_param("selected_threshold", SELECTED_THRESHOLD)
        mlflow.log_param("threshold_selection_reason", "Retention use case prioritizing recall")
        mlflow.log_param("test_size", TEST_SIZE)
        mlflow.log_param("random_state", RANDOM_STATE)

        classifier = best_model.named_steps["classifier"]
        for param_name, param_value in classifier.get_params().items():
            mlflow.log_param(f"classifier__{param_name}", param_value)

        for metric_name, metric_value in final_metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
        )

        mlflow.log_dict(threshold_info, "selected_threshold_90d.json")

        for artifact_path in [MODEL_RESULTS_PATH, RISK_SCORES_PATH, THRESHOLD_RESULTS_PATH]:
            if artifact_path.exists():
                mlflow.log_artifact(str(artifact_path))

        final_run_id = mlflow.active_run().info.run_id

    print("Logged final tuned model to MLflow.")
    print("Run ID:", final_run_id)
else:
    final_run_id = None
    print("LOG_FINAL_MODEL_TO_MLFLOW=False, so no final model was logged to MLflow.")

## 15. Modeling Summary

In [ ]:
modeling_summary = pd.DataFrame(
    [
        {
            "item": "best_model",
            "value": best_model_name,
        },
        {
            "item": "selected_threshold",
            "value": SELECTED_THRESHOLD,
        },
        {
            "item": "test_recall",
            "value": final_metrics["recall"],
        },
        {
            "item": "test_precision",
            "value": final_metrics["precision"],
        },
        {
            "item": "test_f1_score",
            "value": final_metrics["f1_score"],
        },
        {
            "item": "test_roc_auc",
            "value": final_metrics["roc_auc"],
        },
        {
            "item": "mlflow_logging_enabled",
            "value": LOG_MODEL_COMPARISON_TO_MLFLOW or LOG_FINAL_MODEL_TO_MLFLOW,
        },
    ]
)
modeling_summary

## Output Readiness

This notebook is ready when:

1. Candidate models train and compare successfully.
2. The selected threshold aligns with the retention use case.
3. `SAVE_OUTPUTS`, `LOG_MODEL_COMPARISON_TO_MLFLOW`, and `LOG_FINAL_MODEL_TO_MLFLOW` remain `False` unless a
   deliberate regeneration or MLflow logging pass is needed.
4. Saved MLflow results are treated as preserved historical runs unless explicitly recreated.

Recommended next notebook: `06_model_explainability.ipynb`